In [42]:
import numpy as np
from typing import Dict

def _int_to_ip(value: int) -> str:
    value &= 0xFFFFFFFF
    return f"{(value >> 24) & 0xFF}.{(value >> 16) & 0xFF}.{(value >> 8) & 0xFF}.{value & 0xFF}"

def generate_traffic(n_flows: int, n_packets: int, zipf_param: float = 1.2, seed: int = 42) -> Dict[str, object]:
    if n_packets < n_flows:
        raise ValueError("n_packets must be >= n_flows to guarantee at least one packet per flow.")

    rng = np.random.default_rng(seed)

    flow_ids = np.arange(n_flows, dtype=np.int64)

    ip_values = []
    seen = set()
    while len(ip_values) < n_flows:
        needed = n_flows - len(ip_values)
        samples = rng.integers(0, 2**32, size=needed * 2, dtype=np.uint64)
        for val in samples:
            ip_int = int(val)
            if ip_int in seen:
                continue
            seen.add(ip_int)
            ip_values.append(ip_int)
            if len(ip_values) == n_flows:
                break

    flow_ips = np.array([_int_to_ip(ip_int) for ip_int in ip_values], dtype=object)

    weights = rng.zipf(zipf_param, n_flows).astype(np.float64)
    weights = np.clip(weights, 0, None)
    weights /= weights.sum()
    weights[-1] = 1.0 - np.sum(weights[:-1])
    weights = np.clip(weights, 0, None)
    weights /= weights.sum()

    packets_ids = np.empty(n_packets, dtype=np.int64)
    packets_ids[:n_flows] = flow_ids

    remaining = n_packets - n_flows
    unique_sampled = np.array([], dtype=np.int64)
    counts = np.array([], dtype=np.int64)
    if remaining > 0:
        sampled_ids = rng.choice(flow_ids, size=remaining, p=weights)
        packets_ids[n_flows:] = sampled_ids
        unique_sampled, counts = np.unique(sampled_ids, return_counts=True)

    rng.shuffle(packets_ids)

    true_counts_ids = np.ones(n_flows, dtype=np.int64)
    if unique_sampled.size:
        true_counts_ids[unique_sampled] += counts

    packets_ips = flow_ips[packets_ids]

    true_counts_ids_map = {int(flow_id): int(true_counts_ids[flow_id]) for flow_id in flow_ids}
    true_counts_ips = {str(flow_ips[flow_id]): int(true_counts_ids[flow_id]) for flow_id in flow_ids}

    flow_table = [
        {"flow_id": int(flow_id), "ip": str(flow_ips[flow_id])}
        for flow_id in flow_ids
    ]

    return {
        "packets_ids": packets_ids,
        "packets_ips": packets_ips,
        "flow_table": flow_table,
        "true_counts_ids": true_counts_ids_map,
        "true_counts_ips": true_counts_ips,
    }

def generate_flows(n_flows: int, n_packets: int, zipf_param: float = 1.2, seed: int = 42):
    traffic = generate_traffic(n_flows, n_packets, zipf_param=zipf_param, seed=seed)
    return (
        traffic["packets_ips"],
        traffic["true_counts_ips"],
        traffic["packets_ids"],
        traffic["true_counts_ids"],
        traffic["flow_table"],
    )

In [43]:
packets, true_counts_ip, packet_ids, true_counts_ids, flow_table = generate_flows(10000, 3000000)


print(f"Generated {len(packets)} packets.")
print(f"Unique flows appearing in trace: {len(true_counts_ids)} out of 10000")
sample_counts = dict(list(true_counts_ip.items())[:5])
print(f"Sample IP counts: {sample_counts}")
sample_id_counts = dict(list(true_counts_ids.items())[:5])
print(f"Sample ID counts: {sample_id_counts}")
first_packets = [{"flow_id": int(flow_id), "ip": str(packets[idx])} for idx, flow_id in enumerate(packet_ids[:5])]
print(f"First 5 packets (flow_id, ip): {first_packets}")
flow_lookup = {entry["flow_id"]: entry["ip"] for entry in flow_table}
top_flows = sorted(true_counts_ids.items(), key=lambda item: item[1], reverse=True)[:5]
top_flow_summary = [{"flow_id": int(flow_id), "ip": flow_lookup[flow_id], "packets": int(count)} for flow_id, count in top_flows]
print(f"Top 5 flows by packets: {top_flow_summary}")

Generated 3000000 packets.
Unique flows appearing in trace: 10000 out of 10000
Sample IP counts: {'22.217.38.136': 1, '198.33.251.205': 1, '167.145.255.193': 1, '112.90.86.97': 1, '110.218.22.36': 1}
Sample ID counts: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1}
First 5 packets (flow_id, ip): [{'flow_id': 8290, 'ip': '109.27.169.223'}, {'flow_id': 8290, 'ip': '109.27.169.223'}, {'flow_id': 8290, 'ip': '109.27.169.223'}, {'flow_id': 8290, 'ip': '109.27.169.223'}, {'flow_id': 8290, 'ip': '109.27.169.223'}]
Top 5 flows by packets: [{'flow_id': 8290, 'ip': '109.27.169.223', 'packets': 2222661}, {'flow_id': 1425, 'ip': '208.140.37.31', 'packets': 626046}, {'flow_id': 2743, 'ip': '242.131.140.213', 'packets': 72072}, {'flow_id': 2082, 'ip': '83.43.66.186', 'packets': 31105}, {'flow_id': 11, 'ip': '249.194.98.237', 'packets': 27631}]


In [44]:
print(true_counts_ids)
print(true_counts_ip)
print(flow_table)

{0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 27631, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 20: 1, 21: 1, 22: 1, 23: 1, 24: 1, 25: 1, 26: 1, 27: 1, 28: 1, 29: 1, 30: 1, 31: 1, 32: 1, 33: 1, 34: 1, 35: 1, 36: 1, 37: 1, 38: 1, 39: 1, 40: 1, 41: 1, 42: 1, 43: 1, 44: 1, 45: 1, 46: 1, 47: 1, 48: 1, 49: 1, 50: 1, 51: 1, 52: 1, 53: 1, 54: 1, 55: 1, 56: 1, 57: 1, 58: 1, 59: 1, 60: 1, 61: 1, 62: 1, 63: 1, 64: 1, 65: 1, 66: 1, 67: 1, 68: 1, 69: 1, 70: 1, 71: 1, 72: 1, 73: 1, 74: 1, 75: 1, 76: 1, 77: 1, 78: 1, 79: 1, 80: 1, 81: 1, 82: 1, 83: 1, 84: 1, 85: 1, 86: 1, 87: 1, 88: 1, 89: 1, 90: 1, 91: 1, 92: 1, 93: 1, 94: 1, 95: 1, 96: 1, 97: 1, 98: 1, 99: 1, 100: 1, 101: 1, 102: 1, 103: 1, 104: 1, 105: 1, 106: 1, 107: 1, 108: 1, 109: 1, 110: 1, 111: 1, 112: 1, 113: 1, 114: 1, 115: 1, 116: 1, 117: 1, 118: 1, 119: 1, 120: 1, 121: 1, 122: 1, 123: 1, 124: 1, 125: 1, 126: 1, 127: 1, 128: 1, 129: 1, 130: 1, 131: 1, 132: 1, 133: 1, 134: 1, 135: 1, 136: 1, 137: 1, 1

Generated 3000000 packets.
Unique flows appearing in trace: 10000 out of 10000
Top 5 Elephant Flows: {0: 625047, 1: 272499, 2: 167345, 3: 118538, 4: 90659}
